# UMR-CPA Equation of State — NeqSim Examples

This notebook demonstrates the **fused UMR-CPA** equation of state in NeqSim
(`SystemUMRCPAEoS`). The model combines three contributions in a single EOS:

1. A **Peng-Robinson** physical term with a **Mathias-Copeman** alpha function.
2. The **Universal Mixing Rule (UMR)** driven by **UNIFAC** group-contribution
   activity coefficients (the same residual term as `UMR-PRU`).
3. A **CPA association** term for hydrogen-bonding compounds (water, glycols,
   alcohols, alkanolamines).

The pressure is the sum of a physical and an association part:

$$P = P_{\text{PR}}(\text{UMR mixing, MC alpha}) + P_{\text{association}}(\text{CPA})$$

Hydrogen bonding is captured by the **CPA association** term, pure-component
vapour pressure by the **Mathias-Copeman** alpha, and the residual physical
non-ideality by **standard UNIFAC groups**. This follows the natural-gas
dehydration model of Tasios, Louli, Skouras, Solbraa & Voutsas,
*Fluid Phase Equilibria* **2025** (DOI 10.1016/j.fluid.2024.114241).

## Contents
1. Loading the model and selecting the mixing rule
2. Pure-component vapour pressure (water vs steam tables)
3. Water content of a natural-gas stream vs temperature and pressure
4. TEG / water liquid density and a TEG dehydration ternary

In [ ]:
# --- NeqSim setup (devtools dev mode, with pip fallback) ---
import os
import sys
from pathlib import Path


def find_neqsim_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd] + list(cwd.parents))
    for candidate in candidates:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    return None


PROJECT_ROOT = find_neqsim_project_root()
if PROJECT_ROOT is not None:
    sys.path.insert(0, str(PROJECT_ROOT / "devtools"))
    from neqsim_dev_setup import neqsim_init, neqsim_classes

    ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print(f"NeqSim loaded via devtools from {PROJECT_ROOT}")
else:
    import jpype
    try:
        import neqsim  # noqa: F401
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    import neqsim  # noqa: F401

    class _NS:
        JClass = staticmethod(jpype.JClass)

    ns = _NS()
    NEQSIM_MODE = "pip"
    print("NeqSim loaded via pip package")

import matplotlib.pyplot as plt
import numpy as np

SystemUMRCPAEoS = ns.JClass("neqsim.thermo.system.SystemUMRCPAEoS")
ThermodynamicOperations = ns.JClass("neqsim.thermodynamicoperations.ThermodynamicOperations")

print(f"NeqSim mode: {NEQSIM_MODE}")
print(f"SystemUMRCPAEoS: {SystemUMRCPAEoS}")

## 1. Loading the model and selecting the mixing rule

`SystemUMRCPAEoS(T, P)` takes temperature in **Kelvin** and pressure in **bara**.
The model **requires** the Huron-Vidal / `UNIFAC_UMRPRU` mixing rule — it is
selected with `setMixingRule("HV", "UNIFAC_UMRPRU")` (a string pair, *not* a
numeric mixing-rule code). Below we build a wet methane stream and run a
two-phase flash.

In [ ]:
fluid = SystemUMRCPAEoS(298.15, 70.0)  # 25 C, 70 bara
fluid.addComponent("methane", 0.98)
fluid.addComponent("water", 0.02)
fluid.setMixingRule("HV", "UNIFAC_UMRPRU")

ops = ThermodynamicOperations(fluid)
ops.TPflash()
fluid.initProperties()  # initialise transport properties before reading them

print(f"Number of phases : {fluid.getNumberOfPhases()}")
for i in range(fluid.getNumberOfPhases()):
    ph = fluid.getPhase(i)
    name = str(ph.getType())
    rho = ph.getDensity("kg/m3")
    xw = ph.getComponent("water").getx()
    print(f"  phase {i} ({name:>8s}): density = {rho:8.2f} kg/m3,  x(water) = {xw:.3e}")

## 2. Pure-component vapour pressure: water vs steam tables

For a pure component the saturation pressure is obtained with a bubble-point
flash. We compare the UMR-CPA prediction for water against steam-table
reference values from 25 °C to 100 °C. The Mathias-Copeman alpha plus the
regressed CPA water parameters reproduce the saturation curve to within a few
percent.

In [ ]:
def pure_psat(component, temperature_K, pressure_guess_bara):
    """Return the bubble-point (saturation) pressure [bara] of a pure component."""
    f = SystemUMRCPAEoS(temperature_K, pressure_guess_bara)
    f.addComponent(component, 1.0)
    f.setMixingRule("HV", "UNIFAC_UMRPRU")
    o = ThermodynamicOperations(f)
    o.bubblePointPressureFlash(False)
    return float(f.getPressure())


# Steam-table saturation pressures (bara)
T_water = np.array([298.15, 313.15, 323.15, 343.15, 363.15, 373.15])
P_steam = np.array([0.03169, 0.07384, 0.12349, 0.31176, 0.70182, 1.01325])
P_guess = np.array([0.03, 0.07, 0.12, 0.31, 0.70, 1.0])

P_model = np.array([pure_psat("water", T, g) for T, g in zip(T_water, P_guess)])
rel_err = 100.0 * np.abs(P_model - P_steam) / P_steam

for T, pm, ps, e in zip(T_water, P_model, P_steam, rel_err):
    print(f"  T = {T - 273.15:5.1f} C : model {pm:8.5f} bara | steam {ps:8.5f} bara | err {e:4.1f} %")
print(f"\nMean absolute relative deviation: {rel_err.mean():.2f} %")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(T_water - 273.15, P_steam, "ko-", label="Steam tables (reference)", markersize=7)
ax.plot(T_water - 273.15, P_model, "r^--", label="UMR-CPA (NeqSim)", markersize=8)
ax.set_xlabel("Temperature [°C]")
ax.set_ylabel("Saturation pressure [bara]")
ax.set_title("Pure water vapour pressure — UMR-CPA vs steam tables")
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 3. Water content of a natural-gas stream

A core dehydration question is *how much water a gas carries* as a function of
temperature and pressure. We saturate a methane stream with water and read the
gas-phase water mole fraction after a two-phase flash. Physically the water
content must **rise with temperature** and **fall with pressure** — both trends
are reproduced below.

In [ ]:
def gas_water_fraction(temperature_K, pressure_bara):
    """Gas-phase water mole fraction of a water-saturated methane stream."""
    f = SystemUMRCPAEoS(temperature_K, pressure_bara)
    f.addComponent("methane", 0.97)
    f.addComponent("water", 0.03)
    f.setMixingRule("HV", "UNIFAC_UMRPRU")
    o = ThermodynamicOperations(f)
    o.TPflash()
    f.init(2)  # phase compositions (init(3) avoided for associating mixtures)
    if f.getNumberOfPhases() < 2:
        return float("nan")
    return float(f.getPhase(0).getComponent("water").getx())


temps_C = np.array([10.0, 20.0, 30.0, 40.0, 50.0])
temps_K = temps_C + 273.15
pressures = [30.0, 70.0, 120.0]

water_grid = {
    P: np.array([gas_water_fraction(T, P) for T in temps_K]) for P in pressures
}

for P in pressures:
    vals = ", ".join(f"{x * 1e6:6.1f}" for x in water_grid[P])
    print(f"  P = {P:5.0f} bara : x(water) in gas [ppm] = {vals}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for P, c in zip(pressures, colors):
    ax.plot(temps_C, water_grid[P] * 1e6, "o-", color=c, label=f"{P:.0f} bara")
ax.set_xlabel("Temperature [°C]")
ax.set_ylabel("Water content of gas [mol ppm]")
ax.set_title("Gas-phase water content vs temperature and pressure (UMR-CPA)")
ax.legend(title="Pressure")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 4. TEG / water liquid and a TEG dehydration ternary

Triethylene glycol (TEG) is the standard physical solvent for gas dehydration.
The UMR-CPA model treats TEG with **standard UNIFAC groups**
(4 × CH₂ + 2 × CH₂O + 2 × OH) plus a 4C CPA association scheme. Here we:

* check the density of a TEG / water liquid across composition, and
* flash a TEG + water + methane ternary at dehydration conditions to confirm
  TEG and water concentrate in the liquid while methane dominates the gas.

In [ ]:
def teg_water_density(x_teg, temperature_K=298.15, pressure_bara=1.0):
    """Liquid density [kg/m3] of a TEG/water mixture at the given TEG mole fraction."""
    f = SystemUMRCPAEoS(temperature_K, pressure_bara)
    f.addComponent("TEG", x_teg)
    f.addComponent("water", 1.0 - x_teg)
    f.setMixingRule("HV", "UNIFAC_UMRPRU")
    o = ThermodynamicOperations(f)
    o.TPflash()
    f.initProperties()
    return float(f.getPhase(f.getNumberOfPhases() - 1).getDensity("kg/m3"))


x_teg_grid = np.array([0.1, 0.3, 0.5, 0.7, 0.9, 1.0])
rho_grid = np.array([teg_water_density(x) for x in x_teg_grid])

for x, r in zip(x_teg_grid, rho_grid):
    print(f"  x(TEG) = {x:.1f} : density = {r:7.2f} kg/m3")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x_teg_grid, rho_grid, "ms-", markersize=8)
ax.axhspan(900, 1130, color="green", alpha=0.08)
ax.set_xlabel("TEG mole fraction [-]")
ax.set_ylabel("Liquid density [kg/m³]")
ax.set_title("TEG / water liquid density at 25 °C, 1 bara (UMR-CPA)")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# TEG dehydration ternary: methane + water + TEG at 25 C, 60 bara
ternary = SystemUMRCPAEoS(298.15, 60.0)
ternary.addComponent("methane", 0.90)
ternary.addComponent("water", 0.04)
ternary.addComponent("TEG", 0.06)
ternary.setMixingRule("HV", "UNIFAC_UMRPRU")

ops3 = ThermodynamicOperations(ternary)
ops3.TPflash()
ternary.init(2)

gas, liq = 0, ternary.getNumberOfPhases() - 1
print(f"Number of phases: {ternary.getNumberOfPhases()}\n")
print(f"{'component':>10s} | {'gas x':>10s} | {'liquid x':>10s}")
print("-" * 38)
for comp in ["methane", "water", "TEG"]:
    xg = ternary.getPhase(gas).getComponent(comp).getx()
    xl = ternary.getPhase(liq).getComponent(comp).getx()
    print(f"{comp:>10s} | {xg:10.3e} | {xl:10.3e}")

## Summary

* `SystemUMRCPAEoS` fuses a **PR physical term**, the **UMR/UNIFAC** mixing rule,
  a **Mathias-Copeman** alpha and a **CPA association** term in one EOS.
* Always select the mixing rule with `setMixingRule("HV", "UNIFAC_UMRPRU")`.
* It reproduces pure water vapour pressure within a few percent, the correct
  temperature/pressure dependence of gas water content, and realistic TEG / water
  liquid densities — making it well suited to **natural-gas dehydration** studies.
* For associating multicomponent mixtures use `init(2)` or `initProperties()`
  after `TPflash()` (the single-phase gᴱ derivative path used by `init(3)` is not
  populated for every system).

**Reference:** Tasios, Louli, Skouras, Solbraa & Voutsas, *Fluid Phase
Equilibria* (2025), DOI 10.1016/j.fluid.2024.114241.